# Vision-Language Model (VLM) Demo

This notebook shows how to use a vision-language model (Qwen2.5-VL) to answer questions about one or more images.

We will go in small steps so it is easier to teach and explain in class.

## What Is a VLM?

A **Vision-Language Model (VLM)** understands both images and text. You give it one or more images plus a text question, and it returns a text answer.

## What Can VLMs Help Us Do?

- Describe what is in an image
- Count or compare visible objects
- Answer visual Q&A questions
- Summarize information from screenshots or photos

## LLM vs VLM (Quick Difference)

- **LLM**: Works mainly with text input and text output
- **VLM**: Works with image + text input and text output

In short: a VLM is like an LLM with visual understanding.

## Step 1: Install required packages
Run this first in Colab.

In [ ]:
!uv pip install qwen-vl-utils[decord]==0.0.8
!uv pip install unsloth


## Step 2: Import libraries and set environment options

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import torch
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

from unsloth import FastVisionModel
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info


## Step 3: Load the vision-language model and processor

In [ ]:
model, processor = FastVisionModel.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    load_in_4bit=False,
    load_in_8bit=True,
    device_map="auto",
)

model.eval();


## Step 4: Define a helper function for image + prompt chat
This function:
- Displays the input image(s)
- Builds a chat message for the model
- Runs generation
- Returns the final text answer

In [ ]:
def chat_with_images(
    image_urls,
    prompt,
    max_new_tokens=128,
):
    if isinstance(image_urls, str):
        image_urls = [image_urls]

    fig, axes = plt.subplots(1, len(image_urls), figsize=(5 * len(image_urls), 5))
    if len(image_urls) == 1:
        axes = [axes]

    for ax, url in zip(axes, image_urls):
        response = requests.get(url)
        img = Image.open(BytesIO(response.content))
        ax.imshow(img)
        ax.axis("off")

    plt.suptitle("Input Image(s)")
    plt.show()

    content = [{"type": "image", "image": url} for url in image_urls]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
        )

    generated_ids_trimmed = [
        out[len(inp):] for inp, out in zip(inputs.input_ids, outputs)
    ]

    return processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
    )[0]

## Step 5: Import widget tools for a simple interactive demo

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import traceback


## Step 6: Define example inputs

In [ ]:
EXAMPLES = {
    "Single-Image": {
        "urls": [
            "https://potatorolls.com/wp-content/uploads/Lumberjack-Breakfast2.jpg"
        ],
        "prompt": "Describe the food on the table. Please also tell me how many eggs the are on the plate?",
        "max_tokens": 256,
    },
    "Multiple-Images": {
        "urls": [
            "https://housing.berkeley.edu/wp-content/uploads/single_side_01-225px.jpg",
            "https://housing.berkeley.edu/wp-content/uploads/single_side_03-225px.jpg",
            "https://housing.berkeley.edu/wp-content/uploads/single_top-225px.jpg",
        ],
        "prompt": "These are 3 pictures of the same room from different angles, how many beds does the room have?",
        "max_tokens": 512,
    },
}


## Step 7: Build UI controls (URLs, prompt, and run button)

In [ ]:
example_dropdown = widgets.Dropdown(
    options=list(EXAMPLES.keys()),
    description="Examples:"
)

load_example_button = widgets.Button(
    description="Load Example",
    button_style="info",
    icon="download"
)


url1 = widgets.Text(
    placeholder="Image URL 1",
    description="Image 1:",
    layout=widgets.Layout(width="80%"),
    value=EXAMPLES["Single-Image"]["urls"][0],
)

url2 = widgets.Text(
    placeholder="Image URL 2 (optional)",
    description="Image 2:",
    layout=widgets.Layout(width="80%")
)

url3 = widgets.Text(
    placeholder="Image URL 3 (optional)",
    description="Image 3:",
    layout=widgets.Layout(width="80%")
)

prompt_box = widgets.Textarea(
    placeholder="Ask a question about the image(s)...",
    description="Prompt:",
    layout=widgets.Layout(width="80%", height="80px"),
    value=EXAMPLES["Multiple-Images"]["prompt"]
)

max_tokens = widgets.IntSlider(
    value=256,
    min=32,
    max=1024,
    step=32,
    description="Max tokens:"
)

run_button = widgets.Button(
    description="Run VLM",
    button_style="success",
    icon="play"
)

output = widgets.Output()


## Step 8: Add button behaviors (load example + run inference)

In [ ]:
def load_example(b):
    example = EXAMPLES[example_dropdown.value]

    urls = example["urls"] + ["", "", ""]  # pad to 3
    url1.value = urls[0]
    url2.value = urls[1]
    url3.value = urls[2]

    prompt_box.value = example["prompt"]
    max_tokens.value = example["max_tokens"]

load_example_button.on_click(load_example)


def run_inference(b):
    with output:
        clear_output()

        print("Running model... Please wait.\n")

        try:
            urls = [u for u in [url1.value, url2.value, url3.value] if u.strip()]

            if len(urls) == 0:
                print("Please enter at least one image URL.")
                return

            result = chat_with_images(
                urls,
                prompt_box.value,
                max_new_tokens=max_tokens.value
            )

            print("\n" + "="*50)
            print("Final Answer:\n")
            display(Markdown(result))

        except Exception:
            print("Error during inference:")
            traceback.print_exc()


run_button.on_click(run_inference)


## Step 9: Display the interactive app

In [ ]:
ui = widgets.VBox([
    widgets.HTML("<h3>Qwen-VL-demo</h3>"),
    example_dropdown,
    load_example_button,
    url1,
    url2,
    url3,
    prompt_box,
    max_tokens,
    run_button,
    output
])

display(ui)


## Step 10: Class activity ideas

1. Start with the built-in examples and explain what the model gets right.
2. Try your own image URLs and ask clear, specific questions.
3. Ask intentionally hard questions to explore model limits and failure cases.